# Taller de Programación(UBA)
## Clasificación: Logit & KNN. Intro a Curvas ROC.

**Objetivo:** Utilizar los modelos de clasificación de logit y vecinos cercanos (KNN). Visualizar la performance de clasificación

Veremos:
- Clasificación: logit & KNN
- Ilustración del Clasificador de Bayes
- Comparación de modelos con métricas de desempeño de clasificación.
- Introducción a la visualización de las curvas ROC


In [ ]:
import os  
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt  
import seaborn as sns
import statsmodels.api as sm 
from ISLP import load_data

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# metricas de desempeño de clasificiacion
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, recall_score 
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
#from sklearn.metrics import plot_roc_curve

from sklearn.metrics import  auc


## Comparación de modelos usando datos del mercado de acciones 

En este ejemplo, vamos a usar datos del [Stock Market S&P](https://islp.readthedocs.io/en/latest/datasets/Weekly.html) libro ISLP. 
Esta base contiene los retornos porcentuales del S&P 500 stock index por 1250 días, desde inicios de 2001 hasta el final de 2005. Para cada fecha, tenemos:
- Lag1, Lag2,..., Lag5: retornos porcentuales de cada uno de los días anteriores.
- Volume: volumen de acciones negociadas (número de acciones diarias negociadas en miles de millones de dólares)
- Today: retorno porcentual de hoy
- Direction: variable binaria que toma valores "Down" y "Up" indicando si el mercado tuvo un retorno positivo o negativo.


In [ ]:
# Cargamos los datos de Smarket.
smarket = load_data('Smarket')
smarket

In [ ]:
smarket.corr(numeric_only=True).round(2) # con la opcion numeric_only=True hacemo que no tenga en cuenta Direction (string)

In [ ]:
colormap = plt.cm.viridis
plt.figure(figsize=(8,8))
plt.title('Correlacion de Pearson entre las Xs', y=1.05, size=15)
sns.heatmap(smarket.iloc[:,:-1].corr(),
            linewidths=0.1,
            vmax=1.0, 
            square=True, 
            cmap=colormap, 
            linecolor='white', 
            annot=True)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4)) # Un tamaño de gráfico más grande suele verse mejor
smarket.plot(y='Volume', linewidth=.5, ax=ax)
ax.set_title('Tendencias del Volumen del Índice S&P 500', fontsize=16)
ax.set_ylabel('Volumen (promedio diario de\nacciones vendidas en billones)', fontsize=12)
ax.set_xlabel('Orden de Fecha', fontsize=12)
plt.tight_layout() # Ajusta automáticamente los márgenes
plt.show()

In [ ]:
print(smarket['Direction'].value_counts().round(2))

In [ ]:
print(smarket.groupby('Direction').mean().round(2))

Vamos a usar los distintos modelos de clasificación para predicir si sube el indice S&P (**'Direction=Up'**) usando los rezagos (lags) 1 a 5 y el Volumen como predictores.

#### Consideración temporal para el entrenamiento y testeo
En los casos en los que nuestra base de datos tienen una dimensión temporal, el tiempo es una variable "ordenadora" de los datos. Por lo tanto, lo logico es *entrenar* nuestros modelos para la selección de complejidad con **algunos años**, y *testear* nuestro "mejor" modelo en los **últimos año**. De nuevo, con una logica similar a antes, queremos usar aproximadamente un 70% 0 80% de los datos para entrenar y un 30% o 20% para testear.

In [ ]:
# Hacemos el split de la base entre train y test:
train = smarket[smarket.Year < 2005]
test = smarket[smarket.Year >= 2005]
    
ytrain = train['Direction']
ytrain = ytrain.replace('Up', 1)
ytrain = ytrain.replace('Down', 0)
print('\n Var dependiente de entrenamiento', ytrain.shape)

Xtrain = train[['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']]
Xtrain = sm.add_constant(Xtrain)
print('\n X de entrenamiento', Xtrain.shape)

ytest = test['Direction']
ytest = ytest.replace('Up', 1)
ytest = ytest.replace('Down', 0)
print('\n Var dependiente de testeo',ytest.shape)

Xtest = test[['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']]
Xtest = sm.add_constant(Xtest)
print('\n X de testeo', Xtest.shape)

In [ ]:
Xtrain

In [ ]:
# Chequeamos entonces las obsservaciones a predecir en cada base
print(ytest.value_counts())

## 1. Modelo de Regresión Logistica con sckit-learn

Algoritmo de clasificación que se usa para predecir la probabilidad de una variable dependiente categórica. El modelo logit predice $P(Y=1)$ como una función de $X$. Se modela la probabilidad de una forma tal que los outputs serán valores entre 0 y 1 para cualquier valor de $X$.


Ahora utilizaremos la función [LogisticRegression()](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

Se pueden proveer muchos parámetros opcionales para esta función:

- **fit_intercept**: Boolean que decide si calcular el intercepto (True) o considerarlo igual a cero (False). Por default es True.
- **penalty**: Se determina se usar algún tipo de regularización (lo veremos mas adelante en el curso). Posibles valores: ‘l1’, ‘l2’, ‘elasticnet’ y 'None' (usaremos esta opción por ahora). El valor por defecto es default es ‘l2’, es decir que se aplica regularización.


In [ ]:
# Regresión logística
logit_model = LogisticRegression(penalty=None).fit(Xtrain,ytrain)



Prediccion de probabilidad $P(Y=1|X)$

In [ ]:
# Probabilidades predichas
y_prob_logit = logit_model.predict_proba(Xtest)[:,1] 

print('promedio de predicción Prob(y=1|X): ', y_prob_logit.mean().round(3))
print('min de predicción Prob(y=1|X): ', y_prob_logit.min().round(3))
print('max de predicción Prob(y=1|X): ', y_prob_logit.max().round(3))

In [ ]:
y_prob_logit

### 1.1. Visualización de efectos marginales en la probabilidad Y=1|X


In [ ]:
# Gráfico de resultados con Volume (manteniendo otras variables constantes)
plt.figure(figsize=(4,2))

# Extraemos la columna Volume
X_volume = Xtest[['Volume']].values

# Ordenamos los valores de Volume
X_volume_sorted = np.sort(X_volume, axis=0).flatten()

# Creamos una matriz con todas las características (7 columnas incluyendo constante)
X_pred = np.zeros((len(X_volume_sorted), 7))
X_pred[:, 0] = 1  # Constante
X_pred[:, 1] = Xtest['Lag1'].mean()  # Lag1 en su media
X_pred[:, 2] = Xtest['Lag2'].mean()  # Lag2 en su media
X_pred[:, 3] = Xtest['Lag3'].mean()  # Lag3 en su media
X_pred[:, 4] = Xtest['Lag4'].mean()  # Lag4 en su media
X_pred[:, 5] = Xtest['Lag5'].mean()  # Lag5 en su media
X_pred[:, 6] = X_volume_sorted       # Volume varía

# Predicciones
y_pred_score_sorted = logit_model.predict_proba(X_pred)[:,1]   

plt.plot(X_volume_sorted, y_pred_score_sorted, color='blue', zorder=10, label='Prob predicha')
plt.scatter(X_volume, ytest, color='orange', zorder=20, marker="|", label='Y obs.')
plt.xlabel('Volume')
plt.ylabel('Prob(y=1|X)')
plt.legend()
plt.show()

*Ejercicio*: Probar con mejorar la visualización de los graficos

### 1.2. Clasificador de Bayes
Ahora, para la predicción de $\hat{Y}=1$ o $\hat{Y}=0$ usando el modelo estimado de regresión logística, debemos usar la regla de decisión de Bayes de $\hat{p}>0.5$  

In [ ]:
# Convertimos las probabilidades en Y con valores 1 o 0 (usando el clasificador de Bayes)
y_pred_logit = np.where(y_prob_logit > 0.50, 1, 0)

print('Proporcion de 1s predichos: ', y_pred_logit.mean().round(3))
print('Proporcion de 1s reales: ', ytest.astype(int).mean().round(3))

In [ ]:
y_pred_logit

In [ ]:
ytest

In [ ]:
# Equivalente a lo anterior es usar predict() (clasifica en 0s y 1s usando el umbral 0.5)
y_pred_2 = logit_model.predict(Xtest)
print(pd.crosstab(index=y_pred_logit, columns=y_pred_2)) #tabla para chequear que la prediccion es igual


### 1.3. Regresión logística usando Statmodels

In [ ]:
# Podemos repetirlo con statsmodels
# Primero agregamos la columna de 1s y hacemos el ajuste
logit_model2 = sm.Logit(ytrain, Xtrain)
result = logit_model2.fit()
print(result.summary2()) 
#También podríamos vn: print(result.summary2().as_latex())

In [ ]:
# Primero agregamos la columna de 1s y hacemos el ajuste
logit_model2 = sm.Logit(ytrain, Xtrain)
result = logit_model2.fit()
print(result.summary2()) 

Ahora, exportamos en una *linda* tabla de resultados los coeficientes y ratio de probabilidades.

In [ ]:
# Guardamos los coeficientes
coeficientes = logit_model.coef_[0]

# Calculamos los odds ratio
odds_ratio = np.exp(coeficientes)

# Guardamos el error estándar
std_error = result.bse  # Excluir el intercepto

# Creamos la tabla
tabla_resultados = pd.DataFrame({
    #'Variable': X.columns esta linea para cuando tengan muchas variables
    'Coeficiente': coeficientes,
    'Std. Error': std_error,
    'Odds Ratio': odds_ratio
})

tabla_resultados.round(2)

## 2. Vecinos Cercanos (k-nearest neighbors, KNN)
[KNeighborsClassifier()](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html?highlight=kneighborsclassifier#sklearn.neighbors.KNeighborsClassifier): Clasificador de vecinos más cercanos

A continuación veremos un ejemplo de clasificación de las flores en la base de datos iris nuevamente y probaremos ajustando el parámetro k (cantidad de vecinos) para obtener el modelo con mayor precisión

Fuente: [MachineLearning — KNN using scikit-learn](https://towardsdatascience.com/knn-using-scikit-learn-c6bed765be75)

In [ ]:
# Vamos a probar con distintos tamaños de k (cantidad de vecinos)
k_range = range(1,10)
scores = {}      # Para guardar la accuracy en un diccionario
scores_list = [] # Para guardar la accuracy en una lista
for k in k_range:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(Xtrain, ytrain)
        y_pred_knn = knn.predict(Xtest)
        scores[k] = accuracy_score(ytest, y_pred_knn)
        scores_list.append(accuracy_score(ytest, y_pred_knn))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Graficamos la precisión en base a la cantidad de vecinos
plt.plot(k_range, scores_list)
plt.xlabel('Value of K for KNN')
plt.ylabel('Testing Accuracy')

Para $K=7$ la precisión es mayor a 0.54, por lo que podemos usar dicho valor.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=7)
knn_model = knn.fit(Xtrain, ytrain)

In [ ]:
# Predecimos las probabilidades
y_prob_knn = knn_model.predict_proba(Xtest)

In [ ]:
# Convertimos las probabilidades en Y con valores 1 o 0 (usando el clasificador de Bayes)
y_pred_knn = np.where(y_prob_knn > 0.50, 1, 0)

print('Proporcion de 1s predichos: ', y_pred_knn.mean().round(3))
print('Proporcion de 1s reales: ', ytest.astype(int).mean().round(3))

### 3. Medidas de precisión 

Dependiendo la prioridad del problema seguramente vamos a querer usar diferentes métricas. Scikit learn tiene muchas métricas que pueden explorar en el módulo [metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)

- Sensitivity o Recall o True Positive Rate: TP rate = TP/P
- Specificity o True Negative Rate: 1 - FP rate = TN/N
- False Positive Rate o False Alarm Rate: FP rate = FP/N
- False Negative Rate: FN rate = FN/P
- Precision o Positive Predicted Value: TP/(TP+FP)
- Accuracy: (TP+TN)/(P+N)

Nota: Cuidado con las traducciones! "Accuracy" lo pueden encontrar traducido como "precisión" y eso puede generar confusión con la medida "precision" (o positive predicted value). Mi sugerencia es traducir "accuracy" como "exactitud".


[Matriz de confusión](https://www.unite.ai/what-is-a-confusion-matrix/)
<center>
<img src="https://www.unite.ai/wp-content/uploads/2019/12/Preventive_Medicine-e1576294312614.png" width="1000">

</center>

#### Repaso: Matriz de Confusión
La matriz de confusión de sklearn pone en las filas las Y reales y las columnas las Y predichas. Muestra así los valores:

                               predicción
                         real   tn fp
                                fn tp

In [ ]:
confusion_matrix(ytest,y_pred_logit)

In [ ]:
# Predicciones de probabilidad Y=1 
y_prob_knn = knn_model.predict_proba(Xtest)[:, 1]
y_prob_logit = logit_model.predict_proba(Xtest)[:, 1]

# reasignamos 1s y 0s segun el umbral de la regla de bayes
y_pred_knn = (y_prob_knn > 0.5).astype(int)
y_pred_logit = (y_prob_logit > 0.5).astype(int)

# Matrices de confusión
cm_knn = confusion_matrix(ytest, y_pred_knn)
cm_logit = confusion_matrix(ytest, y_pred_logit)

# Calcular métricas
def calcular_metricas(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Tasa de Verdadero Positivo (Sensibilidad)
    tpr = tp / (tp + fn)
    
    # Tasa de Falso Positivo
    fpr = fp / (fp + tn)
    
    # AUC
    auc_score = roc_auc_score(y_true, y_prob)
    
    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)
    
    return tpr, fpr, auc_score, accuracy

# Calcular métricas para KNN
tpr_knn_bayes, fpr_knn_bayes, auc_knn, acc_knn = calcular_metricas(ytest, y_pred_knn, y_prob_knn)

# Calcular métricas para Logit
tpr_logit_bayes, fpr_logit_bayes, auc_logit, acc_logit = calcular_metricas(ytest, y_pred_logit, y_prob_logit)

# Crear tabla de comparación
tabla_comparacion = pd.DataFrame({
    'Métrica': ['Tasa Verdadero Positivo (Sensibilidad)', 
                'Tasa Falso Positivo', 
                'AUC', 
                'Accuracy'],
    'KNN': [round(tpr_knn_bayes, 3), 
            round(fpr_knn_bayes, 3), 
            round(auc_knn, 3), 
            round(acc_knn, 3)],
    'Logit': [round(tpr_logit_bayes, 3), 
              round(fpr_logit_bayes, 3), 
              round(auc_logit, 3), 
              round(acc_logit, 3)]
})

print(tabla_comparacion)

### Curva ROC                  
ROC: Receiver Operating Characteristics

Veremos como utilizar las funciones:

-  [roc_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html#sklearn.metrics.roc_curve): computa la curva de ROC
- [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html#sklearn.metrics.roc_auc_score): Computa el area bajo la curva de ROC de los scores predichos.
- [RocCurveDisplay](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html#sklearn.metrics.RocCurveDisplay): Sirve para visualizar la curva de ROC. Con el mismo fin existe [plot_roc_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.plot_roc_curve.html#sklearn.metrics.plot_roc_curve)

Para estas funciones necesitamos predecir dicha probabilidad condicional $p=Pr(Y=1|X)$ o *score*

In [ ]:
# Calculamos las curvas ROC
fpr_knn, tpr_knn, _ = roc_curve(ytest, y_prob_knn)
roc_auc_knn = auc(fpr_knn, tpr_knn)

fpr_logit, tpr_logit, _ = roc_curve(ytest, y_prob_logit)
roc_auc_logit = auc(fpr_logit, tpr_logit)

# Graficar
plt.figure(figsize=(8, 6))
plt.plot(fpr_knn, tpr_knn, color='blue', lw=2, label=f'KNN (AUC = {roc_auc_knn:.3f})')
plt.plot(fpr_logit, tpr_logit, color='red', lw=2, label=f'Logit (AUC = {roc_auc_logit:.3f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Clasificador Aleatorio')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falso Positivo')
plt.ylabel('Tasa de Verdadero Positivo')
plt.title('Curva ROC - Comparación KNN vs Logit')
plt.legend(loc="lower right")
plt.show()

